# Spin chain with one- and two-magnon manifolds

This example describes a finite periodic spin chain truncated at two spin flips. It demonstrates:

- one shared ground state for all one-magnon transitions;
- excitation manifolds of different dimensions;
- transitions from the ground state to one magnon and from one to two magnons;
- linear spectroscopy of the one-magnon band;
- ground-state-bleach, stimulated-emission, and excited-state-absorption contributions to a third-order rephasing spectrum; and
- numerical agreement and scaling differences between the dense-Liouville and sparse-sector backends.

Natural units are used, with $\hbar=1$. Energies are in eV and times are in eV$^{-1}$.

In [ ]:
from itertools import combinations
from pathlib import Path
from time import perf_counter
import sys

import matplotlib.pyplot as plt
import numpy as np

# Locate the project root when the notebook is launched from examples/
# or from another directory inside the repository.
current_directory = Path.cwd().resolve()
for candidate in (current_directory, *current_directory.parents):
    if (candidate / "projet_solver10.py").is_file():
        project_root = str(candidate)
        if project_root not in sys.path:
            sys.path.insert(0, project_root)
        break
else:
    raise RuntimeError("Could not locate the directory containing projet_solver10.py.")

from projet_solver10 import (
    ExcitationSectorModel,
    FrequencyPathway,
    PropagationInterval,
    SpectroscopyPlotter,
    SpectroscopyProtocol,
    SpectroscopySolver,
    standard_nq_protocol,
)

## 1. Periodic spin chain and excitation manifolds

We use a number-conserving spin-$1/2$ chain with periodic boundary conditions,

$$
H=\omega_0\sum_j n_j
+J\sum_j\left(S_j^+S_{j+1}^-+S_j^-S_{j+1}^+\right)
+U\sum_j n_jn_{j+1}.
$$

The Hilbert space is truncated after two spin flips,

$$\mathcal H=\mathcal H_0\oplus\mathcal H_1\oplus\mathcal H_2,$$

with dimensions $1$, $L$, and $\binom{L}{2}$. The state $|G\rangle=|\downarrow\cdots\downarrow\rangle$ is the unique state in $\mathcal H_0$. All one-magnon eigenstates are excitations of this same state; the ground state is not duplicated for every momentum.

The nearest-neighbor interaction $U$ shifts configurations in which two spin flips occupy adjacent sites. For $U=0$, the two-magnon manifold provides the noninteracting reference. A nonzero $U$ modifies the two-magnon resonances without changing the one-magnon dispersion.

In [ ]:
def excitation_basis(n_sites, n_excitations):
    return tuple(combinations(range(n_sites), n_excitations))


def build_sector_hamiltonian(
    n_sites, n_excitations, omega_0, exchange, interaction
):
    basis = excitation_basis(n_sites, n_excitations)
    state_index = {state: index for index, state in enumerate(basis)}
    hamiltonian = np.zeros((len(basis), len(basis)), dtype=complex)

    for column, state in enumerate(basis):
        occupied = set(state)
        adjacent_pairs = sum(
            (site + 1) % n_sites in occupied for site in occupied
        )
        hamiltonian[column, column] = (
            n_excitations * omega_0 + interaction * adjacent_pairs
        )

        for site in state:
            for neighbor in ((site - 1) % n_sites, (site + 1) % n_sites):
                if neighbor in occupied:
                    continue
                target = tuple(sorted((occupied - {site}) | {neighbor}))
                row = state_index[target]
                hamiltonian[row, column] += exchange

    assert np.allclose(hamiltonian, hamiltonian.conj().T)
    return basis, hamiltonian


def build_raising_block(source_basis, target_basis, probe_profile):
    target_index = {state: index for index, state in enumerate(target_basis)}
    block = np.zeros((len(target_basis), len(source_basis)), dtype=complex)
    for column, state in enumerate(source_basis):
        occupied = set(state)
        for site, amplitude in enumerate(probe_profile):
            if site in occupied or amplitude == 0:
                continue
            target = tuple(sorted((*state, site)))
            block[target_index[target], column] += amplitude
    return block


def localized_probe_profile(n_sites, probe_site, probe_width):
    sites = np.arange(n_sites)
    clockwise = (sites - probe_site) % n_sites
    counterclockwise = (probe_site - sites) % n_sites
    distance = np.minimum(clockwise, counterclockwise)
    profile = np.exp(-0.5 * (distance / probe_width) ** 2)
    return profile / np.linalg.norm(profile)


def make_spin_chain_model(
    n_sites, omega_0, exchange, interaction, *, probe_site=0, probe_width=0.8
):
    bases = {}
    hamiltonians = {}
    for n_excitations in (0, 1, 2):
        bases[n_excitations], hamiltonians[n_excitations] = (
            build_sector_hamiltonian(
                n_sites, n_excitations, omega_0, exchange, interaction
            )
        )

    # A spatially localized probe has broad momentum content. Its finite width
    # also lets two successive raising interactions act on different sites.
    probe_profile = localized_probe_profile(
        n_sites, probe_site, probe_width
    ).astype(complex)
    raising_blocks = {
        (1, 0): build_raising_block(bases[0], bases[1], probe_profile),
        (2, 1): build_raising_block(bases[1], bases[2], probe_profile),
    }

    model = ExcitationSectorModel(
        hamiltonians,
        raising_blocks,
        initial_sector=0,
    )
    return model, bases, hamiltonians, raising_blocks

## 2. One- and two-magnon spectra

In the one-excitation sector, the Hamiltonian has the cosine dispersion

$$\omega(k)=\omega_0+2J\cos k.$$

The two-excitation sector contains $\binom{L}{2}$ states. Its transition energies enter the third-order signal through coherences between $\mathcal H_2$ and $\mathcal H_1$. The local interaction term $U$ affects only this second manifold and therefore separates one-magnon band effects from bimagnon interactions.

In [ ]:
n_sites = 6
omega_0 = 1.55
exchange = -0.08
bimagnon_interaction = -0.12
eta = 0.02

model, bases, hamiltonians, raising_blocks = make_spin_chain_model(
    n_sites, omega_0, exchange, bimagnon_interaction
)

one_magnon_energies = np.linalg.eigvalsh(hamiltonians[1])
two_magnon_energies = np.linalg.eigvalsh(hamiltonians[2])
k_points = 2.0 * np.pi * np.arange(n_sites) / n_sites
expected_one_magnon = omega_0 + 2.0 * exchange * np.cos(k_points)

assert np.allclose(
    np.sort(one_magnon_energies), np.sort(expected_one_magnon), atol=1e-12
)
assert model.dimension(0) == 1
assert model.dimension(1) == n_sites
assert model.dimension(2) == n_sites * (n_sites - 1) // 2
assert model.initial_condition().sector == 0

print("Sector dimensions:", {sector: model.dimension(sector) for sector in model.sectors()})
print(f"Total Hilbert dimension: {sum(model.dimension(s) for s in model.sectors())}")
print(
    f"One-magnon band: {one_magnon_energies.min():.4f} to "
    f"{one_magnon_energies.max():.4f} eV"
)
print(
    f"Two-magnon energies: {two_magnon_energies.min():.4f} to "
    f"{two_magnon_energies.max():.4f} eV"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), constrained_layout=True)
axes[0].plot(k_points, expected_one_magnon, "o-")
axes[0].set_xlabel(r"Momentum $k$")
axes[0].set_ylabel("One-magnon energy (eV)")
axes[0].set_title("One-magnon dispersion")

axes[1].plot(
    np.arange(two_magnon_energies.size),
    np.sort(two_magnon_energies),
    "o",
    markersize=4,
)
axes[1].set_xlabel("Two-magnon eigenstate index")
axes[1].set_ylabel("Two-magnon energy (eV)")
axes[1].set_title(rf"Two-magnon manifold, $U={bimagnon_interaction:.2f}$ eV")

## 3. Excitation-sector model and backend agreement

`ExcitationSectorModel` diagonalizes each excitation manifold separately and transforms every rectangular transition block as

$$\widetilde J_{ts}=U_t^\dagger J_{ts}U_s.$$

The raising operator contains the blocks $J_{10}:\mathcal H_0\rightarrow\mathcal H_1$ and $J_{21}:\mathcal H_1\rightarrow\mathcal H_2$. Their adjoints are generated automatically. Both numerical backends receive exactly the same model. The dense backend materializes the Liouville generator, whereas the sparse backend applies it as a matrix-free linear operator.

In [ ]:
sparse_solver = SpectroscopySolver(
    backend="sparse_sector", eta=eta, krylov_tolerance=1e-11
)
sparse_solver.feed_model(model)

dense_solver = SpectroscopySolver(
    backend="dense_liouville", eta=eta, cache_resolvents=False
)
dense_solver.feed_model(model)

assert sparse_solver.summary()["total_dimension"] == dense_solver.summary()["total_dimension"]
assert model.transition_decomposition() == "explicit_sector"

print("Sparse backend:", sparse_solver.summary()["backend"])
print("Dense backend:", dense_solver.summary()["backend"])
print("Sectors:", sparse_solver.summary()["sector_dimensions"])
print("Transition decomposition:", model.transition_decomposition())

## 4. Linear response of the one-magnon band

A single ket-side raising interaction creates a coherence $|1\mathrm{mag}\rangle\langle G|$. The spatially localized probe has broad momentum content, so the linear spectrum samples the complete one-magnon band. Its finite spatial width also allows two successive raising interactions to act on different sites. Dense and sparse calculations are evaluated on the same frequency grid and must agree point by point.

In [ ]:
linear_pathway = FrequencyPathway(
    name="linear",
    interactions=("Ku",),
    component="linear",
)
linear_protocol = SpectroscopyProtocol(
    intervals=(PropagationInterval("omega", "frequency", coherence_order=1),),
    name="linear_frequency",
)

omega = np.linspace(1.30, 1.80, 81)


def calculate_linear_response(solver):
    return np.asarray(
        [
            solver.calc_pathway(
                linear_pathway, linear_protocol, {"omega": frequency}
            ).value
            for frequency in omega
        ]
    )


linear_sparse = calculate_linear_response(sparse_solver)
linear_dense = calculate_linear_response(dense_solver)
linear_backend_residual = np.max(np.abs(linear_sparse - linear_dense))

assert np.all(np.isfinite(linear_sparse))
assert np.allclose(linear_sparse, linear_dense, rtol=1e-8, atol=1e-9)
print(f"Dense--sparse linear residual: {linear_backend_residual:.3e}")

In [ ]:
fig, axes = plt.subplots(
    2, 1, figsize=(7.2, 5.7), sharex=True, constrained_layout=True
)
axes[0].plot(omega, np.abs(linear_dense), label="dense", linewidth=2.2)
axes[0].plot(omega, np.abs(linear_sparse), "--", label="sparse")
for energy in one_magnon_energies:
    axes[0].axvline(energy, color="0.75", linewidth=0.7)
axes[0].set_ylabel(r"$|P^{(1)}(\omega)|$")
axes[0].set_title("Local-probe one-magnon response")
axes[0].legend()
axes[1].semilogy(omega, np.maximum(np.abs(linear_sparse - linear_dense), 1e-18))
axes[1].set_xlabel(r"Energy $\omega$ (eV)")
axes[1].set_ylabel("Absolute difference")

## 5. Third-order rephasing response and bimagnon ESA

The ground-state-bleach and stimulated-emission pathways are

$$R_{\mathrm{GSB}}=(B_u,B_d,K_u),\qquad R_{\mathrm{SE}}=(B_u,K_u,B_d).$$

The two-magnon manifold adds the excited-state-absorption pathway

$$R_{\mathrm{ESA}}=(B_u,K_u,K_u).$$

All three pathways have coherence history $q=(-1,0,+1)$. In the ESA pathway, the final ket-side raising interaction maps $\mathcal H_1$ into $\mathcal H_2$, leaving a $|2\mathrm{mag}\rangle\langle1\mathrm{mag}|$ emission coherence. Its pathway prefactor has the opposite sign to the GSB and SE contributions generated by the commutator convention.

In [ ]:
pathway_se = FrequencyPathway(
    name="SE",
    interactions=("Bu", "Ku", "Bd"),
    component="rephasing",
)
pathway_gsb = FrequencyPathway(
    name="GSB",
    interactions=("Bu", "Bd", "Ku"),
    component="rephasing",
)
pathway_esa = FrequencyPathway(
    name="ESA",
    interactions=("Bu", "Ku", "Ku"),
    component="rephasing",
)

protocol_1q = standard_nq_protocol(
    order=1,
    nq_interval=1,
    detection_interval=3,
    n_interactions=3,
    nq_axis="omega_1q",
    detection_axis="omega_emit",
)

omega_1q = np.linspace(-1.80, -1.30, 17)
omega_emit = np.linspace(1.30, 1.80, 17)
result_rephasing = sparse_solver.generate_spectrum(
    protocol_1q,
    axes={"omega_1q": omega_1q, "omega_emit": omega_emit},
    fixed_coordinates={"t2": 0.0},
    pathways=(pathway_gsb, pathway_se, pathway_esa),
)

gsb = result_rephasing.pathways["GSB"]
se = result_rephasing.pathways["SE"]
esa = result_rephasing.pathways["ESA"]
rephasing = result_rephasing.components["rephasing"]
reconstruction_residual = np.max(np.abs(rephasing - (gsb + se + esa)))
reference_amplitude = max(np.max(np.abs(gsb)), np.max(np.abs(se)))

assert np.all(np.isfinite(rephasing))
assert np.max(np.abs(esa)) > 1e-6 * reference_amplitude
assert reconstruction_residual < 1e-10

peak_index = np.unravel_index(np.argmax(np.abs(rephasing)), rephasing.shape)
print(f"Pathway reconstruction residual: {reconstruction_residual:.3e}")
print(f"Maximum ESA amplitude: {np.max(np.abs(esa)):.6g}")
print("Strongest total rephasing coordinate:")
print(f"  omega_1q   = {omega_1q[peak_index[0]]:.4f} eV")
print(f"  omega_emit = {omega_emit[peak_index[1]]:.4f} eV")

In [ ]:
plotter = SpectroscopyPlotter(detection_phase=0.0)

pathway_plot = plotter.plot_spectrum_result(
    result_rephasing,
    params={
        "source": "pathways",
        "names": ["GSB", "SE", "ESA"],
        "view": "abs",
        "normalization": "global",
        "labels": (
            r"Emission energy $\omega_{\mathrm{emit}}$ (eV)",
            r"Excitation energy $\omega_{1Q}$ (eV)",
        ),
        "title": r"Pathway-resolved $\chi^{(3)}$ rephasing response",
        "diagonals": "auto",
        "style": {"abs_cmap": "magma", "levels": 30, "contour_lines": False},
    },
)

total_plot = plotter.plot_spectrum_result(
    result_rephasing,
    params={
        "source": "components",
        "names": ["rephasing"],
        "view": "all",
        "normalization": "row",
        "labels": (
            r"Emission energy $\omega_{\mathrm{emit}}$ (eV)",
            r"Excitation energy $\omega_{1Q}$ (eV)",
        ),
        "title": r"Total spin-chain $\chi^{(3)}$ rephasing spectrum",
        "diagonals": "auto",
        "style": {"cmap": "RdYlBu_r", "abs_cmap": "magma", "levels": 30, "contour_lines": False},
    },
)

## 6. Dense--sparse scaling

For a chain truncated at two excitations,

$$D(L)=1+L+\binom{L}{2}.$$

The dense backend stores Liouville matrices with $D^4$ complex entries. The sparse backend does not materialize these matrices, although a mixed-state or frequency-domain calculation still carries density vectors with $D^2$ entries. We time backend construction and one linear-response point for $L=4,\ldots,8$. Memory curves are structural estimates for one dense Liouville matrix and one sparse density vector; caches and temporary solver workspaces require additional memory.

In [ ]:
def benchmark_backend(benchmark_model, backend_name):
    options = (
        {"cache_resolvents": False}
        if backend_name == "dense_liouville"
        else {"krylov_tolerance": 1e-11}
    )
    solver = SpectroscopySolver(backend=backend_name, eta=eta, **options)
    start = perf_counter()
    solver.feed_model(benchmark_model)
    build_time = perf_counter() - start

    start = perf_counter()
    value = solver.calc_pathway(
        linear_pathway, linear_protocol, {"omega": omega_0}
    ).value
    solve_time = perf_counter() - start
    return build_time, solve_time, value


benchmark_rows = []
for length in range(4, 9):
    benchmark_model, _, _, _ = make_spin_chain_model(
        length, omega_0, exchange, bimagnon_interaction
    )
    dimension = sum(benchmark_model.dimension(s) for s in benchmark_model.sectors())
    dense_build, dense_solve, dense_value = benchmark_backend(
        benchmark_model, "dense_liouville"
    )
    sparse_build, sparse_solve, sparse_value = benchmark_backend(
        benchmark_model, "sparse_sector"
    )
    assert np.allclose(sparse_value, dense_value, rtol=1e-8, atol=1e-9)
    benchmark_rows.append(
        {
            "L": length,
            "D": dimension,
            "dense_build": dense_build,
            "dense_solve": dense_solve,
            "sparse_build": sparse_build,
            "sparse_solve": sparse_solve,
            "residual": abs(sparse_value - dense_value),
        }
    )

print(" L    D   dense build  sparse build  dense solve  sparse solve  residual")
for row in benchmark_rows:
    print(
        f"{row['L']:2d}  {row['D']:3d}   {row['dense_build']:10.4f}  "
        f"{row['sparse_build']:11.4f}  {row['dense_solve']:10.4f}  "
        f"{row['sparse_solve']:11.4f}  {row['residual']:.2e}"
    )

In [ ]:
timed_lengths = np.asarray([row["L"] for row in benchmark_rows])
dense_total_times = np.asarray(
    [row["dense_build"] + row["dense_solve"] for row in benchmark_rows]
)
sparse_total_times = np.asarray(
    [row["sparse_build"] + row["sparse_solve"] for row in benchmark_rows]
)

memory_lengths = np.arange(4, 13)
memory_dimensions = 1 + memory_lengths + memory_lengths * (memory_lengths - 1) // 2
complex_bytes = np.dtype(np.complex128).itemsize
dense_matrix_mib = complex_bytes * memory_dimensions**4 / 2**20
sparse_vector_mib = complex_bytes * memory_dimensions**2 / 2**20

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.0), constrained_layout=True)
axes[0].semilogy(timed_lengths, dense_total_times, "o-", label="dense")
axes[0].semilogy(timed_lengths, sparse_total_times, "o-", label="sparse")
axes[0].set_xlabel("Number of sites $L$")
axes[0].set_ylabel("Build + one-point time (s)")
axes[0].set_title("Measured runtime")
axes[0].legend()

axes[1].semilogy(memory_lengths, dense_matrix_mib, "o-", label=r"dense matrix $D^4$")
axes[1].semilogy(memory_lengths, sparse_vector_mib, "o-", label=r"sparse vector $D^2$")
axes[1].set_xlabel("Number of sites $L$")
axes[1].set_ylabel("Structural memory (MiB)")
axes[1].set_title("Dominant stored object")
axes[1].legend()

## 7. What this example establishes

1. `ExcitationSectorModel` represents manifolds of unequal dimensions and transforms rectangular operators between their eigenbases.
2. A unique ground state can couple coherently to the complete one-magnon band.
3. The same raising operator connects the one-magnon and two-magnon manifolds.
4. The two-magnon manifold produces a finite excited-state-absorption pathway that is absent from an independent two-level description.
5. Dense and sparse backends reproduce the same response for sizes accessible to both.
6. The dense Liouville representation has a $D^4$ structural memory cost, while the matrix-free sparse calculation avoids materializing that matrix.

The model remains an exact-diagonalization calculation truncated at two spin flips. It does not include three-magnon pathways, dissipative scattering, or the thermodynamic limit. The timing comparison is an executable scaling demonstration rather than a hardware-independent performance claim.